In [4]:
# --------------------------------------------------------------
#  Imports
# --------------------------------------------------------------
from sage.all import *
import sage.libs.lrcalc.lrcalc as lrcalc


In [5]:
def degree(p):
    """|p| = sum of its parts."""
    return sum(p)


def generate_partitions(n, max_part=None):
    """Yield all partitions of n as tuples in non‑increasing order."""
    if n == 0:
        yield ()
    else:
        if max_part is None or max_part > n:
            max_part = n
        for first in range(max_part, 0, -1):
            for rest in generate_partitions(n - first, first):
                yield (first,) + rest


def subset_partitions(p):
    """All partitions whose degree is ≤ |p| (returned as a list)."""
    d = degree(p)
    res = []
    for n in range(d + 1):
        res.extend(list(generate_partitions(n)))
    return res


In [6]:
print("degree((3,2,1)) =", degree((3,2,1)))       # → 6
print("partitions of 3 :", list(generate_partitions(3)))
print("subset of (2,1):", subset_partitions((2,1)))


degree((3,2,1)) = 6
partitions of 3 : [(3,), (2, 1), (1, 1, 1)]
subset of (2,1): [(), (1,), (2,), (1, 1), (3,), (2, 1), (1, 1, 1)]


In [7]:
def solveOne(k, k1, k2, k3, k4):
    """
    Solve
        2·γ + δ + p + q = k                 (1)
        k1 − k2 = γ + δ + p                 (2)
        k3 − k4 = γ + δ + q                 (3)

    together with
        0 ≤ δ,γ,p,q < k1  and  0 ≤ δ,γ,p,q < k3
        k2 < k1 , k4 < k3   (checked here)

    Returns a list of dictionaries
        {"δ":δ, "γ":γ, "p":p, "q":q, "deg λ'":k2, "deg μ'":k4}
    """
    if not (k2 < k1 and k4 < k3):
        return []                     # illegal input

    # δ = (k1−k2)+(k3−k4)−k
    delta = k1 - k2 + k3 - k4 - k
    if delta < 0:
        return []                     # δ must be ≥ 0

    # maximal allowed (δ+γ)
    max_sum = min(k1 - k2, k3 - k4)
    if delta > max_sum:
        return []                     # no room for a non‑negative γ

    sols = []
    max_gamma = max_sum - delta       # γ ∈ [0, max_gamma]
    for gamma in range(max_gamma + 1):
        s   = delta + gamma           # s = δ+γ
        p   = (k1 - k2) - s
        q   = (k3 - k4) - s
        if p < 0 or q < 0:
            continue
        # each variable must be < k1 and < k3
        if not (delta < k1 and gamma < k1 and p < k1 and q < k1):
            continue
        if not (delta < k3 and gamma < k3 and p < k3 and q < k3):
            continue
        sols.append({
            "δ": delta,
            "γ": gamma,
            "p": p,
            "q": q,
            "deg λ'": k2,
            "deg μ'": k4
        })
    return sols


In [8]:
solveOne(1,2,1,2,1)

[{'δ': 1, 'γ': 0, 'p': 0, 'q': 0, "deg λ'": 1, "deg μ'": 1}]

In [9]:
# Example: λ = (3,2) → |λ| = 5
#          μ = (3,2) → |μ| = 5
#          λ′ = (2,1) → deg = 3
#          μ′ = (2,1) → deg = 3
k  = 2                 # the constant that appears in (1)
k1 = 5                 # |λ|
k2 = 3                 # |λ'|
k3 = 5                 # |μ|
k4 = 3                 # |μ'|

sols = solveOne(k, k1, k2, k3, k4)
print(sols)


[{'δ': 2, 'γ': 0, 'p': 0, 'q': 0, "deg λ'": 3, "deg μ'": 3}]


In [10]:

def _lrcoef(p, q, r):
    """
    Littlewood‑Richardson coefficient  c^{r}_{p,q}  with proper handling of
    the empty partition.

    *If p or q is the empty partition () the coefficient is 1 iff the
    remaining (non‑empty) partition equals r; otherwise we delegate to the
    high‑level Sage function LR, which works for all shapes.*

    Parameters
    ----------
    p, q, r : partitions written as tuples.
              The empty partition must be the empty tuple   ().

    Returns
    -------
    int – the LR coefficient.
    """
    # both first arguments empty
    if p == () and q == ():
        return int(r == ())

    # exactly one of the first two is empty
    if p == ():
        return int(q == r)          # c^{r}_{∅,q}
    if q == ():
        return int(p == r)          # c^{r}_{p,∅}

    # generic (non‑empty) case – use Sage’s full implementation
    return int(lrcalc.lrcoef(p, q, r))


In [11]:
print(_lrcoef((), (3,), (3,)))        # 1
print(_lrcoef((2,), (), (2,)))        # 1
print(_lrcoef((2,), (1,1), (3,1)))   # 1   (a genuine non‑trivial case)
print(_lrcoef((2,), (1,1), (4,)))    # 0


1
1
0
0


In [12]:
def _deg_to_partition(d):
    """0 → ();  d>0 → (d,)  (a one‑part partition)"""
    return () if d == 0 else (d,)


def lrcoef4(mu1, mu2, mu3, mu4, lam):
    """
    Three‑fold convolution

        Σ_{a ⊢ (mu1+mu2)} Σ_{b ⊢ (mu3+mu4)}
               c^{a}_{mu1,mu2} · c^{b}_{mu3,mu4} · c^{lam}_{a,b}

    mu1,…,mu4 are *degrees* (non‑negative integers).  ``lam`` must be the
    **full partition** λ (a tuple, e.g. (3,2)).
    """
    total1 = mu1 + mu2                # degree of the first intermediate partition
    total2 = mu3 + mu4                # degree of the second intermediate partition

    s = 0
    for a in generate_partitions(total1):
        c1 = _lrcoef(_deg_to_partition(mu1),
                      _deg_to_partition(mu2), a)
        if c1 == 0:
            continue
        for b in generate_partitions(total2):
            c2 = _lrcoef(_deg_to_partition(mu3),
                          _deg_to_partition(mu4), b)
            if c2 == 0:
                continue
            c3 = _lrcoef(a, b, lam)
            if c3 == 0:
                continue
            s += c1 * c2 * c3
    return s


In [13]:
lam = (3, 2)          # the full partition λ
deg_lprime = 3        # |λ′| = 3  (this will be passed as mu4)

# the call that previously gave 0
print("lrcoef4(2,0,0,3,lam) =",
      lrcoef4(2, 0, 0, deg_lprime, lam))


lrcoef4(2,0,0,3,lam) = 0
